# 02 - Preprocessing and augmentation

How an X-ray becomes a model input, and what augmentation does to it.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make the project's `src` package importable from inside notebooks/
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), Path.cwd())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src import config
from src.utils import find_dataset_root, count_images_per_class

# Point this at data/dummy to try the notebook without the real dataset
DATA_DIR = config.DATA_DIR
ROOT_DATA = find_dataset_root(DATA_DIR)
print("Dataset root:", ROOT_DATA)

import keras
from src.data import build_augmentation
from src.predict import preprocess_image

## From file to tensor

Grayscale X-rays are converted to 3 channels (pretrained backbones expect RGB) and resized to a square. Pixel values stay in 0-255; each model rescales them internally (see `src/models.py`).

In [ ]:
path = sorted((ROOT_DATA / "train" / "PNEUMONIA").glob("*"))[0]
original = Image.open(path)
display_img, batch = preprocess_image(path, config.IMG_SIZE)

print("original  : size", original.size, "mode", original.mode)
print("model input:", batch.shape, batch.dtype, "range", batch.min(), "-", batch.max())

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(original.convert("L"), cmap="gray"); axes[0].set_title(f"Original {original.size}")
axes[1].imshow(display_img); axes[1].set_title(f"Resized {config.IMG_SIZE}x{config.IMG_SIZE}, 3 channels")
axes[2].hist(display_img[..., 0].ravel(), bins=50); axes[2].set_title("Pixel intensity histogram")
for ax in axes[:2]: ax.axis("off")
plt.tight_layout(); plt.show()

## Augmentation

Random rotation, zoom, shift and contrast change are applied to *training* images only. Horizontal flips are intentionally left out because they would put the heart on the wrong side.

In [ ]:
augment = build_augmentation()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(display_img); axes[0, 0].set_title("Original"); axes[0, 0].axis("off")
for ax in list(axes.ravel())[1:]:
    aug = augment(batch, training=True).numpy()[0]
    ax.imshow(np.clip(aug, 0, 255).astype("uint8")); ax.set_title("Augmented"); ax.axis("off")
plt.tight_layout(); plt.show()

## Why a validation split from `train/`?

The Kaggle `val/` folder only has 16 images - far too few to trust early stopping or threshold selection. `src/data.py` keeps 15% of `train/` for validation and leaves `test/` untouched until the final evaluation.